# Data Extraction

## Skin-Deep Insights: Aspect-Based Sentiment Analysis of Sephora Reviews

**Goal:** Create a structured working dataset by selecting only the columns
required for ABSA. This stage does NOT clean data — it isolates the right
features and saves a working file separate from the raw source.

**Why extraction is a separate stage from cleaning:**
Mixing extraction with cleaning creates a messy, hard-to-debug pipeline.
Extraction defines WHAT we work with. Cleaning defines HOW we prepare it.
Keeping them separate makes the project reproducible and auditable.

**Approach:**
```
Load combined reviews → Select 8 business-relevant columns
→ Convert date format → Validate integrity → Save working_reviews.csv
```
**Columns selected and justification:**

| Column | Type | Reason |
|---|---|---|
| product_id | Identifier | Links reviews to products |
| product_name | Text | Human-readable product reference |
| brand_name | Categorical | Brand-level segmentation |
| price_usd | Numeric | Price tier analysis |
| rating | Numeric | Core comparison for hidden dissatisfaction |
| review_text | Text | Primary NLP analysis target |
| skin_type | Categorical | Demographic segmentation |
| review_date | DateTime | Time-based trend analysis |


**Load Data**

In [1]:
import pandas as pd
reviews = pd.read_csv("../sephora/reviews.csv")  

/var/folders/_4/sclc4l610q36b3bg5c1xkksh0000gn/T/ipykernel_58329/1492613641.py:2: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews = pd.read_csv("../sephora/reviews.csv")


In [2]:
reviews.shape

(1094411, 21)

Raw dataset confirmed: 1,094,411 reviews, 21 columns. We will reduce
this to 8 columns — retaining only what is necessary for ABSA and
segmentation analysis. 13 columns are dropped because they either
introduce privacy concerns (author_id), are not relevant to aspect
analysis (eye_color, hair_color), or belong to a separate engagement
analysis scope (helpfulness, feedback counts).

In [3]:
reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1094411 entries, 0 to 1094410
Data columns (total 21 columns):
 #   Column                    Non-Null Count    Dtype  
---  ------                    --------------    -----  
 0   Unnamed: 0                1094411 non-null  int64  
 1   author_id                 1094411 non-null  object 
 2   rating                    1094411 non-null  int64  
 3   is_recommended            926423 non-null   float64
 4   helpfulness               532819 non-null   float64
 5   total_feedback_count      1094411 non-null  int64  
 6   total_neg_feedback_count  1094411 non-null  int64  
 7   total_pos_feedback_count  1094411 non-null  int64  
 8   submission_time           1094411 non-null  object 
 9   review_text               1092967 non-null  object 
 10  review_title              783757 non-null   object 
 11  skin_tone                 923872 non-null   object 
 12  eye_color                 884783 non-null   object 
 13  skin_type                 9

In [4]:
reviews['review_date'] = pd.to_datetime(reviews['review_date'])
print(reviews['review_date'].dtype)

datetime64[ns]


In [5]:
working_df = reviews[[
    'product_id',
    'product_name',
    'brand_name',
    'price_usd',
    'rating',
    'review_text',
    'skin_type',
    'review_date'
]].copy()

working_df.shape

(1094411, 8)

Dimensionality reduced from 21 to 8 columns. All dropped columns are
either analytically irrelevant to this project scope or better addressed
in a dedicated separate analysis.

In [6]:
working_df = working_df.drop(columns=['Unnamed: 0'], errors='ignore')

In [7]:
working_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1094411 entries, 0 to 1094410
Data columns (total 8 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   product_id    1094411 non-null  object        
 1   product_name  1094411 non-null  object        
 2   brand_name    1094411 non-null  object        
 3   price_usd     1094411 non-null  float64       
 4   rating        1094411 non-null  int64         
 5   review_text   1092967 non-null  object        
 6   skin_type     982854 non-null   object        
 7   review_date   1094411 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(1), object(5)
memory usage: 66.8+ MB


In [8]:
working_df.isnull().sum()

product_id           0
product_name         0
brand_name           0
price_usd            0
rating               0
review_text       1444
skin_type       111557
review_date          0
dtype: int64

**Post-Extraction Null Assessment:**

| Column | Nulls | Decision |
|---|---|---|
| review_text | 1,444 (0.13%) | Remove in Stage 5 — NLP requires text |
| skin_type | 111,557 (10.2%) | Retain as NaN — never impute demographics |
| All others | 0 | Structurally complete |

Demographic data (skin_type) should not be imputed. Guessing a user's
skin type would introduce artificial signal into segment analysis. NaN
is the honest representation of missing demographic data.

In [9]:
working_df.duplicated().sum()

np.int64(608)

608 full-row duplicates detected. We do not blindly drop them here — duplicate
inspection requires understanding whether they are data errors or
legitimate repeated opinions before deletion.

In [10]:
working_df.to_csv("../sephora/working_reviews.csv", index=False)

# Summary — Data Extraction

**What was completed in this stage:**

The raw 1,094,411-row dataset was reduced from 21 columns to 8
business-relevant features. Date formatting was applied and data
integrity was verified post-extraction.

| Metric | Value |
|---|---|
| Input rows | 1,094,411 |
| Input columns | 21 |
| Output rows | 1,094,411 (unchanged — no data altered) |
| Output columns | 8 |
| Missing review_text | 1,444 (0.13%) |
| Missing skin_type | 111,557 (10.2%) |
| Duplicates detected | 608 |

**Output file:** `working_reviews.csv`

**Key principle:** Extraction does not alter data. It selects and
structures. Original review content is untouched — only the scope
has been reduced to what is analytically relevant for ABSA.

**Next:** Stage 5 — Data Cleaning will handle nulls, duplicates,
short reviews, incentivized reviews, and text normalization.